In [1]:
# Run this cell if you haven't installed the packages yet
# !pip install pandas numpy matplotlib scikit-learn xgboost optuna joblib

import pandas as pd
import numpy as np
import hashlib
import os
from datetime import datetime, timedelta

print("✅ Libraries imported")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")

✅ Libraries imported
   pandas  : 2.3.3
   numpy   : 2.2.6


In [2]:
np.random.seed(42)

N          = 50_000
FRAUD_RATE = 0.015

def hash_id(val):
    return hashlib.md5(val.encode()).hexdigest()[:12]

# Time range: full year 2024
start      = datetime(2024, 1, 1)
timestamps = [start + timedelta(seconds=int(s))
              for s in np.sort(np.random.randint(0, 365*24*3600, N))]

is_fraud = np.random.binomial(1, FRAUD_RATE, N)

merchant_cats = ["grocery","food","travel","electronics","clothing",
                 "pharmacy","fuel","entertainment","jewelry","gaming"]
device_types  = ["mobile","web","pos","atm"]
channels      = ["online","in-store","contactless","manual"]
countries     = ["IN","US","GB","SG","AE","CN","DE","FR"]
cities        = ["Mumbai","Delhi","Bangalore","Chennai","Kolkata",
                 "Pune","Hyderabad","Jaipur","Surat","Lucknow"]

# Fraud → higher amounts
amount = np.where(
    is_fraud == 1,
    np.random.lognormal(mean=8.5, sigma=1.5, size=N),
    np.random.lognormal(mean=6.0, sigma=1.2, size=N)
).round(2)

merchant_cat = np.random.choice(merchant_cats, N)

# Electronics / Jewelry → more fraud
for cat, boost in [("electronics", 0.6), ("jewelry", 0.5), ("gaming", 0.4)]:
    mask = (merchant_cat == cat)
    is_fraud[mask] = np.where(
        np.random.random(mask.sum()) < boost,
        np.random.binomial(1, FRAUD_RATE * 5, mask.sum()),
        is_fraud[mask]
    )

hour      = np.array([t.hour for t in timestamps])
dayofweek = np.array([t.weekday() for t in timestamps])
is_night  = (hour < 6) | (hour > 22)

# Night → more fraud
is_fraud[is_night] = np.where(
    np.random.random(is_night.sum()) < 0.05,
    1, is_fraud[is_night]
)

is_international = np.random.binomial(1, 0.08, N).astype(bool)

# International → more fraud
is_fraud[is_international] = np.where(
    np.random.random(is_international.sum()) < 0.06,
    1, is_fraud[is_international]
)

# Velocity features (fraudsters do many transactions quickly)
prev_24h_tx_count_card = np.where(
    is_fraud == 1,
    np.random.poisson(8, N),
    np.random.poisson(2, N)
).astype(float)

prev_24h_amt_card     = (prev_24h_tx_count_card * amount * np.random.uniform(0.3, 1.2, N)).round(2)
prev_1h_tx_count_card = np.where(
    is_fraud == 1,
    np.random.poisson(4, N),
    np.random.poisson(0.5, N)
).astype(float)
velocity_amt_1h = (prev_1h_tx_count_card * amount * np.random.uniform(0.5, 2.0, N)).round(2)

n_cards, n_merchants = 5000, 2000
card_ids     = [hash_id(f"card_{np.random.randint(n_cards)}") for _ in range(N)]
merchant_ids = [hash_id(f"merch_{np.random.randint(n_merchants)}") for _ in range(N)]

df_raw = pd.DataFrame({
    "tx_id":                  [f"TX{str(i).zfill(7)}" for i in range(N)],
    "ts":                     [t.strftime("%Y-%m-%d %H:%M:%S") for t in timestamps],
    "amount":                 amount,
    "merchant_cat":           merchant_cat,
    "merchant_id_hash":       merchant_ids,
    "card_id_hash":           card_ids,
    "city":                   np.random.choice(cities, N),
    "country":                np.random.choice(countries, N),
    "device_type":            np.random.choice(device_types, N),
    "channel":                np.random.choice(channels, N),
    "hour":                   hour,
    "dayofweek":              dayofweek,
    "prev_24h_tx_count_card": prev_24h_tx_count_card,
    "prev_24h_amt_card":      prev_24h_amt_card,
    "prev_1h_tx_count_card":  prev_1h_tx_count_card,
    "velocity_amt_1h":        velocity_amt_1h,
    "is_international":       is_international,
    "is_night":               is_night.astype(bool),
    "is_fraud":               is_fraud.astype(int),
})

os.makedirs("data", exist_ok=True)
df_raw.to_csv("data/transactions.csv", index=False)
print(f"✅ Generated {N:,} transactions")
print(f"   Fraud rate : {df_raw.is_fraud.mean():.2%}")
print(f"   Saved to   : data/transactions.csv")

✅ Generated 50,000 transactions
   Fraud rate : 4.29%
   Saved to   : data/transactions.csv


In [3]:
SCHEMA = {
    "tx_id":                   "string",
    "ts":                      "string",
    "amount":                  "float64",
    "merchant_id_hash":        "string",
    "card_id_hash":            "string",
    "hour":                    "int64",
    "dayofweek":               "int64",
    "prev_24h_tx_count_card":  "float64",
    "prev_24h_amt_card":       "float64",
    "prev_1h_tx_count_card":   "float64",
    "velocity_amt_1h":         "float64",
    "is_international":        "bool",
    "is_night":                "bool",
    "is_fraud":                "int64",
}

df = pd.read_csv("data/transactions.csv")

# Apply schema (skip category columns — handle separately)
for col, dtype in SCHEMA.items():
    df[col] = df[col].astype(dtype)

df["ts"] = pd.to_datetime(df["ts"])
df = df.sort_values("ts").reset_index(drop=True)

# Save clean version
df.to_csv("data/transactions_clean.csv", index=False)

print(f"✅ Ingested and cleaned")
print(f"   Shape : {df.shape}")
print(f"   Fraud : {df.is_fraud.mean():.2%}")
print()
print(df.dtypes)

✅ Ingested and cleaned
   Shape : (50000, 19)
   Fraud : 4.29%

tx_id                     string[python]
ts                        datetime64[ns]
amount                           float64
merchant_cat                      object
merchant_id_hash          string[python]
card_id_hash              string[python]
city                              object
country                           object
device_type                       object
channel                           object
hour                               int64
dayofweek                          int64
prev_24h_tx_count_card           float64
prev_24h_amt_card                float64
prev_1h_tx_count_card            float64
velocity_amt_1h                  float64
is_international                    bool
is_night                            bool
is_fraud                           int64
dtype: object
